In [ ]:
%%html
<style>
.jp-Notebook {
    max-width: 800px;
    margin: 0 auto;
    padding: 0 40px;
}

.jp-Cell {
    max-width: 800px;
    margin-left: auto;
    margin-right: auto;
}

.jp-MarkdownCell .jp-RenderedMarkdown {
    font-size: 15px;
    line-height: 1.7;
}
</style>

SyntaxError: invalid decimal literal (2752426096.py, line 3)

# **Points of Interest as Crime Generators: A Supervised Regression Analysis of London Ward-Level Crime Density**

> **Student Number:** 25127081
>
> I, Emily Deeb, pledge my honour that the work presented in this assessment is my own. Where information has been derived from other sources, I confirm that this has been indicated in the work.

## **Preparation**

- [Github link](google.com) *[Optional]*

- Number of words: ***

- Runtime: *** hours (*Memory 10 GB, CPU Intel i7-10700 CPU @2.90GHz*)

- Coding environment: SDS Docker (or anything else)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/) (or other license that you like).

- Additional library *[libraries not included in SDS Docker or not used in this module]*:
    - **watermark**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.
    - ......

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

---

## **Introduction**

[[ go back to the top ]](#Table-of-contents)

This study examines the spatial relationship between urban amenity density and crime concentration across London wards. We use open-source point-of-interest (POI) data from OpenStreetMap and recorded crime data from the Metropolitan Police Service within a supervised regression model to assess whether the spatial layout of urban amenities such as cafés, clubs, and restaurants predicts crime density and whether specific amenity types are associated with different crime categories.

Routine Activity Theory (Cohen and Felson, 1979) argues that crime occurs when motivated offenders, suitable targets, and the absence of capable guardians intersect in the same place and time. This convergence can be defined by the functional land-use profile of an area; the mix of amenities, services, and commercial activity that determines who is present and when. Brantingham and Brantingham (1995) formalise this with the concept of crime generators, theorising that locations that attract large numbers of people for non-criminal reasons increase crime opportunities by providing many potential offenders and targets. 

Point-of-interest (POI) data has become an important resource in empirical research on urban crime. Cichosz (2020) demonstrated that OSM POI density features are highly effective at predicting crime risk across UK urban areas, surpassing traditional land-use classifications. POI data also serves as a proxy for ambient population: Andresen (2011) argued that the number of people actually present, rather than the residential population, is a more meaningful denominator for calculating crime rates. This methodological distinction is crucial, as Malleson and Andresen (2016) showed for London that relying on residential population data generates misleading hotspots for non-residential crimes such as theft from the person.

Although significant progress has been made in analyzing the relationship between crime and urban fabric, most studies continue to model total crime or individual crime types in isolation. The extent to which different POI categories generate distinct crime types, or whether urban density drives crime uniformly across categories, remains underexplored, especially at the ward level in London. To address this gap, we split the crime dataset into two types; Type A (person/society crime) and Type B (property crime), and analysed these alongside POI categories. This study is among the first to apply SHAP decomposition to compare crime generator profiles across multiple crime categories simultaneously at ward level in London.

---

## **Research question**

[[ go back to the top ]](#Table-of-contents)

**Primary:** To what extent does the density of urban amenities predict person/society and property crime densities across London wards?

**Sub-question:** Which specific amenity categories serve as the primary crime generators for each crime type, and do these differ systematically between person/society and property crime?

----

## **Data and Feature Engineering**

[[ go back to the top ]](#Table-of-contents)

Three datasets were integrated for this analysis. 

**Data 01: Metropolitan Police Service recorded crime data** were obtained from the London Datastore (data.london.gov.uk), containing monthly crime counts by ward and crime category for the 24 months to February 2025. Crimes were summed across the full period and aggregated into two theoretically motivated categories: 


**Table 1:** Crime category definitions and offence types included in the analysis

| Crime Category | Type | Offence Types Included |
|:---|:---|:---|
| Type A — Person/Society Crime | Target variable | Violence against the person, robbery, sexual offences, possession of weapons, drug offences, public order offences, miscellaneous crimes against society |
| Type B — Property Crime | Target variable | Theft, burglary, vehicle offences, arson and criminal damage |

*Note: NFIB fraud offences were excluded from both categories as fraud is recorded at the victim's home address rather than the location of the offence, rendering it spatially uninformative at the ward level.*

**Data 02: Ward Spatial Boundary and population**

**Table 2:** Spatial and demographic data sources

| Dataset | Source | URL | Geography | Notes |
|:---|:---|:---|:---|:---|
| Ward boundary polygons | ONS Open Geography Portal | geoportal.statistics.gov.uk | Ward level | 2024 boundaries, filtered to London using GLA boundary |
| Ward areas | Derived from boundary polygons | — | Ward level | Calculated using British National Grid projection (EPSG:27700) |
| Population estimates | ONS mid-2021 ward-level estimates | geoportal.statistics.gov.uk | Ward level | No missing values after merging; 679 London wards retained |

*Note: Population density was retained in the dataset but excluded from model features to isolate the contribution of amenity configuration independently of population size. Incorporating it as a control variable is a natural next step.*

**Data 03: Point-of-interest data** were extracted from OpenStreetMap using the osmnx Python library (Boeing, 2017) and queried within the Greater London Authority boundary polygon. After cleaning, 28,059 POIs were retained across 18 categories. The table below shows the categories, OSM tags used for extraction, and counts:

**Table 3:** Point-of-interest categories, OSM extraction tags, and counts

| POI Category | OSM Tag | Count |
|:---|:---|---:|
| Restaurant | amenity=restaurant | 5,938 |
| Cafe | amenity=cafe | 4,637 |
| Fast food | amenity=fast_food | 4,132 |
| Convenience store | shop=convenience | 3,584 |
| Clothes shop | shop=clothes | 1,826 |
| ATM | amenity=atm | 1,674 |
| Pub | amenity=pub | 1,278 |
| Pharmacy | amenity=pharmacy | 886 |
| Bar | amenity=bar | 877 |
| Parking | amenity=parking | 708 |
| Supermarket | shop=supermarket | 669 |
| Rail station | railway=station | 616 |
| Bank | amenity=bank | 431 |
| Jewellery shop | shop=jewelry | 429 |
| Electronics shop | shop=electronics | 182 |
| Nightclub | amenity=nightclub | 110 |
| Department store | shop=department_store | 71 |
| Mall | shop=mall | 11 |
| **Total** |  -> | **28,059** |

*Source: OpenStreetMap via osmnx Python library (Boeing, 2017), extracted within the Greater London Authority boundary polygon.*

**Feature Engineering:** All crime and POI counts were normalised by ward area to produce density measures (per km²), accounting for variation in ward size across the 679 London wards. A log transformation was then applied to all density variables to address right skewness prior to modelling:

$$x' = \log\left(\frac{\text{count}_i}{\text{area}_i} + 1\right)$$

where $x'$ is the transformed feature, $\text{count}_i$ is the raw crime or POI count for ward $i$, $\text{area}_i$ is the ward area in km², and the $+1$ constant prevents undefined values where counts are zero. After transformation, residual skewness fell below 1.5 for 14 of the 18 POI features, with only mall, department store, nightclub, and jewellery remaining moderately skewed due to their sparse distribution across wards.

**Table 4:** Variable definitions, sources, units, and roles in the analysis

| Variable | Type | Source | Unit | Role |
|:---|:---|:---|:---|:---|
| typeA_density | Continuous | MPS / London Datastore | crimes per km² | Target — Type A |
| typeB_density | Continuous | MPS / London Datastore | crimes per km² | Target — Type B |
| restaurant_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| cafe_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| fast_food_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| convenience_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| clothes_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| atm_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| pub_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| pharmacy_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| bar_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| parking_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| supermarket_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| station_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| bank_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| jewelry_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| electronics_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| nightclub_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| department_store_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| mall_density | Continuous | OpenStreetMap | POIs per km² | Feature |
| area_km2 | Continuous | ONS / EPSG:27700 | km² | Normalisation denominator |
| population | Continuous | ONS mid-2021 | persons | Context variable |

*Note: All log-transformed versions of the above density variables (suffix \_log) are used as model inputs. The final modelling dataset comprises 679 wards × 18 features with no missing values.*

---

## **Methodology**
[[ go back to the top ]](#Table-of-contents)

The methodology is structured as a progression of analytical lenses. We begin with Multiple Linear Regression, which assumes the simplest possible relationship between POI density and crime; a straight line that assumes each amenity type contributes independently and additively. This gives us a readable baseline that deliberately oversimplifies reality. We then step back and ask a different question with K-Nearest Neighbors: rather than fitting a line, do wards with similar amenity profiles experience similar crime levels? If so, feature-space similarity alone should be predictive. Lastly, Random Forest removes all these assumptions, building hundreds of decision trees across random subsets of the data to capture the complex, non-linear interactions between amenity types that the previous two methods cannot. Having established what the model predicts, SHAP values evaluate it from the inside, decomposing each prediction into the individual contributions of each amenity type, showing not just which features matter, but how much and in which direction. Together, these four analyses give us a description, a prediction, and an interpretation.

> The dataset (679 wards × 18 features) was split 80/20 into training (543 wards) and test (136 wards) sets using a fixed random seed (42) for reproducibility.

1. **MLR** — Features were standardised prior to fitting to enable direct coefficient comparison. A relevant limitation here is multicollinearity, as café and restaurant density are strongly correlated (r = 0.85), which can produce unstable coefficient estimates.
2. **KNN** — The optimal number of neighbors (k=11) was selected via 5-fold cross-validation on the training set.
3. **RF** — Hyperparameters were tuned via 5-fold cross-validated grid search, yielding n_estimators=300, max_depth=10, and min_samples_leaf=4 (CV R²=0.683).
4. **SHAP** — Values were computed from the tuned RF models at the ward level. Unlike global feature importance measures, SHAP values are additive and locally consistent, enabling direct comparison of which amenity types drive each crime category and in which direction.

**Table 5:** Model performance on held-out test set (R² and RMSE) for Type A and Type B crime density

| Method | Type A R² | Type A RMSE | Type B R² | Type B RMSE |
|:---|:---:|:---:|:---:|:---:|
| Multiple Linear Regression | 0.521 | 0.591 | 0.630 | 0.585 |
| K-Nearest Neighbors (k=11) | 0.404 | 0.659 | 0.535 | 0.655 |
| **Random Forest** | **0.635** | **0.516** | **0.714** | **0.514** |

*Note: R² and RMSE calculated on the held-out test set (136 wards). Bold indicates the best-performing model. Random Forest was selected as the basis for SHAP interpretation.*

----

## **Results and Discussion**

[[ go back to the top ]](#Table-of-contents)

**Model performance**

As shown in Table X, Random Forest outperformed both baseline methods across all targets and metrics. The consistent gap between RF and MLR indicates that the POI-crime relationship is non-linear, with amenity types interacting in ways that additive linear models cannot fully capture. This is consistent with Cichosz (2020), who found that random forest significantly outperformed logistic regression and decision trees across all UK urban areas and crime types. The underperformance of KNN suggests that wards with similar POI profiles do not always exhibit similar crime levels, pointing to spatial heterogeneity in crime determinants across London's diverse ward typologies.

Property crime (Type B) was predicted more accurately than person/society crime (Type A) across all three methods. This supports the theoretical distinction between the two crime types: property crime is more directly opportunity-driven and therefore more strongly determined by the physical configuration of targets, whereas person/society crime is additionally influenced by social and behavioural factors not captured by POI data alone.

MLR coefficients showed negative values for restaurant, cafe, and pub in the Type A model — an artefact of multicollinearity rather than a genuine inverse relationship, as confirmed by their positive SHAP values in the RF model. 

**Crime generators by type**

SHAP analysis demonstrates that different amenity categories are primary drivers for each crime type. For Type A (person/society crime), convenience store density is the dominant predictor (maximum SHAP value: 0.75), followed by fast food and ATM density. Notably, traditional night-time economy venues — bars, pubs, and nightclubs — were among the weakest predictors despite their prominent role in criminological discourse. This may reflect the mandatory guardianship associated with licensed premises — door staff, CCTV, and licensing conditions — which suppresses crime opportunity despite high footfall, consistent with the capable guardians component of Routine Activity Theory (Cohen and Felson, 1979). This finding also suggests that sustained daytime footfall generates more person/society crime opportunity in London wards than night-time activity, consistent with Andresen's (2011) ambient population argument.

For Type B (property crime), café density was the primary predictor (maximum SHAP value: 1.25), followed by convenience stores and restaurants. The shift from fast food (Type A) to café (Type B) as the dominant predictor suggests that functionally distinct spatial profiles underlie each crime category. Fast food outlets tend to cluster in high-footfall deprived areas where violence concentrates, while cafés cluster in daytime commercial zones where theft and shoplifting are more prevalent.

**Spatial diagnostics**

Spatial residuals reveal systematic geographic patterns in model error. Underpredicted wards — where actual crime exceeds model estimates — cluster in deprived outer London boroughs such as Barnet, Brent, Enfield, and Croydon. These areas show elevated crime despite sparse amenity infrastructure, suggesting that socioeconomic deprivation drives crime independently of physical land use. Conversely, overpredicted wards cluster in affluent outer boroughs including Bromley, Havering, and Kingston, where high amenity density does not correspond to elevated crime — likely reflecting the crime-suppressing effects of high property values and private security. The spatial structure in the residuals confirms that POI density captures opportunity-driven crime concentration effectively but underestimates crime driven by socioeconomic factors.

## **Conclusion**

[[ go back to the top ]](#Table-of-contents)

This study demonstrates that the spatial configuration of urban amenities, measured entirely from open-source POI data, predicts crime density across London wards with moderate to strong accuracy — RF R² of 0.635 for person/society crime and 0.714 for property crime — without recourse to demographic or socioeconomic variables. The key finding is that amenity type matters as much as amenity density. SHAP analysis revealed that different POI categories drive distinct crime types, a distinction not captured by studies that model single crime outcomes. Convenience stores and fast food outlets are the primary generators of person/society crime, while cafés dominate property crime prediction. Night-time economy venues, despite their prominence in criminological literature, showed unexpectedly weak predictive power — a pattern consistent with the guardianship effects of licensed premises regulation.

The spatial residuals point to the boundary of what POI data can explain. Where the model fails systematically — underpredicting in deprived outer boroughs and overpredicting in affluent ones — socioeconomic factors appear to operate independently of physical land use. Zhou et al. (2023) demonstrate that socioeconomic characteristics explain significant additional spatial variance in London ward-level crime rates (adjusted R²=0.36). Taken together, these two frameworks are complementary: POI density captures the physical opportunity structure of crime, while socioeconomic variables capture the social conditions that generate it. Future work combining both feature sets could provide a more complete picture of crime determinants across London's diverse urban landscape.

Limitations include the dark figure of crime in police-recorded data, which likely compounds underprediction in deprived wards; the ecological fallacy inherent in ward-level aggregation; incomplete OSM coverage in outer London; and the absence of temporal, demographic, and socioeconomic controls.

## **References**

[[ go back to the top ]](#Table-of-contents)